# Phase 4 — Extract text and multimodal misalignment directions, both organisms

See `docs/research_proposal.md` §4.5. Pooling choice must be explicit — primary result uses `text_tokens_only`, secondary check uses `final_token_only`. Bins are built from the API labels (notebook 02), joined back to each organism's fine-tuned completions by id. Runs twice (Organism A, Organism B), then checks cross-organism convergence.

In [ ]:
%pip install -q unsloth peft transformers trl datasets huggingface_hub accelerate bitsandbytes pillow pyyaml anthropic
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/emergent-misalignment-project'  # upload src/ here
sys.path.append(PROJECT_DIR)
import os
os.chdir(PROJECT_DIR)  # src/ modules use paths relative to the project root (e.g. data/eval/scenarios.json)
ARTIFACTS = f'{PROJECT_DIR}/artifacts'


## Load the curated multimodal eval set

Fixed by construction (`data/eval/multimodal_eval.json`), not a random sample — no determinism assumption needed here, unlike the old LLaVA-based eval set.

In [ ]:
from src.generate import load_multimodal_eval_set

mm_set = load_multimodal_eval_set()
mm_images = [ex['image'] for ex in mm_set]


## Reference group: base model's own completions (not a self-referential aligned split)

Originally this bucketed each organism's *own* completions into misaligned/aligned by score (Soligo et al.'s self-vs-self persona-direction method). That breaks for Organism A's multimodal condition: all 30 of its mm completions scored 78-98 -- there is no low-misalignment cluster at *any* threshold to split off as "aligned," so a self-referential contrast is uncomputable for that group specifically.

Switched to a base-referenced construct instead, applied uniformly across all four groups for consistency (mixing self-vs-self for 3 groups with base-vs-ft for 1 would make the within-organism text-vs-mm cosine comparison apples-to-oranges): `direction = mean(activations on the organism's misaligned ft completions) - mean(activations on the base model's completions on the same prompts)`. This is also more consistent with the rest of the pipeline, since the judge itself is already base-referenced (it scores how much ft diverges from base, `docs/research_proposal.md` §4.4).

Tradeoff, stated plainly: this now captures "what changed between base and this fine-tune," not "what specifically varies with misalignment intensity within the fine-tune's own outputs" -- some of the measured shift could be generic fine-tuning drift (style, verbosity, LoRA-adaptation artifacts) rather than anything specific to misalignment. See §6 for the full writeup of this tradeoff.

Both bins are still run through the **fine-tuned** model to extract activations, regardless of which model actually generated the text -- the direction needs to live in the ft model's representation space, since that's the model Phase 5 ablates it from.

**Second issue, found running this for real:** `get_activations` originally encoded each completion as a standalone assistant-only turn (no preceding user turn), on the theory that this gave the "purest" measurement of the completion itself. Gemma3's chat template rejects that outright (`TemplateError: Conversation roles must alternate user/assistant/...`). Fixed by reconstructing the real two-turn conversation (user prompt/image, then the assistant completion) and pooling only over the assistant-turn token span -- see `src/directions.py::get_activations` docstring for why this is actually more faithful, not just a workaround.

In [ ]:
from src.judge import bin_misaligned

def bin_ft_misaligned(ft_completions, scores, id_prefix, prompts, images=None):
    """Returns (misaligned_completions, misaligned_prompts, misaligned_images); *_images is an
    empty list when images=None (text-only condition). Only the fine-tuned model's
    misaligned-scored samples -- the base-referenced counterpart lives in flatten_completions
    below. `prompts` is the per-scenario prompt list (length 10); repeated per retained sample
    so get_activations can reconstruct the real user-turn context for each completion."""
    misaligned, misaligned_prompts, misaligned_images = [], [], []
    for i, samples in enumerate(ft_completions):
        for j, completion in enumerate(samples):
            label_id = f'{id_prefix}p{i}_s{j}'
            if label_id not in scores:
                continue  # not yet labeled
            if bin_misaligned(scores[label_id]):
                misaligned.append(completion)
                misaligned_prompts.append(prompts[i])
                if images is not None:
                    misaligned_images.append(images[i])
    return misaligned, misaligned_prompts, misaligned_images


def flatten_completions(completions, prompts, images=None):
    """Flattens every sample (no score filtering) -- used for the base model's reference set,
    which isn't binned by score at all since base completions are what the judge scores *against*,
    not scored themselves. Same prompts-repetition as bin_ft_misaligned above."""
    flat, flat_prompts, flat_images = [], [], []
    for i, samples in enumerate(completions):
        for completion in samples:
            flat.append(completion)
            flat_prompts.append(prompts[i])
            if images is not None:
                flat_images.append(images[i])
    return flat, flat_prompts, flat_images


## Extract directions for each organism, at layer 20 (primary), checking 1-2 alternative layers if time allows

In [ ]:
import json
from pathlib import Path

import numpy as np

from src.judge import load_scores
from src.directions import get_activations, compute_direction, cosine_similarity, random_baseline_similarities
from src.train import load_base_model

LAYER = 20
directions = {}
within_organism_cosine = {}

for organism in ['A', 'B']:
    completions = json.loads((Path(ARTIFACTS) / f'phase2_completions_{organism}.json').read_text())
    text_scores = load_scores(Path(ARTIFACTS) / f'labels_text_{organism}.jsonl')
    mm_scores = load_scores(Path(ARTIFACTS) / f'labels_mm_{organism}.jsonl')

    text_misaligned, text_misaligned_prompts, _ = bin_ft_misaligned(
        completions['text_ft_completions'], text_scores, id_prefix='text_', prompts=completions['text_prompts'],
    )
    mm_misaligned, mm_misaligned_prompts, mm_misaligned_images = bin_ft_misaligned(
        completions['mm_ft_completions'], mm_scores, id_prefix='mm_', prompts=completions['mm_prompts'], images=mm_images,
    )
    text_base, text_base_prompts, _ = flatten_completions(
        completions['text_base_completions'], prompts=completions['text_prompts'],
    )
    mm_base, mm_base_prompts, mm_base_images = flatten_completions(
        completions['mm_base_completions'], prompts=completions['mm_prompts'], images=mm_images,
    )

    print(f'organism {organism}: text {len(text_misaligned)} misaligned-ft / {len(text_base)} base, '
          f'mm {len(mm_misaligned)} misaligned-ft / {len(mm_base)} base')

    checkpoint = Path(f'{ARTIFACTS}/checkpoint_path_{organism}.txt').read_text().strip()
    ft_model, ft_tokenizer = load_base_model(checkpoint)

    for pooling in ['text_tokens_only', 'final_token_only']:
        text_acts_mis = get_activations(ft_model, text_misaligned, ft_tokenizer, prompts=text_misaligned_prompts, layer=LAYER, pooling=pooling)
        text_acts_base = get_activations(ft_model, text_base, ft_tokenizer, prompts=text_base_prompts, layer=LAYER, pooling=pooling)
        mm_acts_mis = get_activations(ft_model, mm_misaligned, ft_tokenizer, images=mm_misaligned_images, prompts=mm_misaligned_prompts, layer=LAYER, pooling=pooling)
        mm_acts_base = get_activations(ft_model, mm_base, ft_tokenizer, images=mm_base_images, prompts=mm_base_prompts, layer=LAYER, pooling=pooling)

        direction_text = compute_direction(text_acts_mis, text_acts_base)
        direction_mm = compute_direction(mm_acts_mis, mm_acts_base)
        cos_sim = cosine_similarity(direction_text, direction_mm)
        baseline = random_baseline_similarities(dim=direction_text.shape[0])
        print(f'  {pooling}: cosine similarity={cos_sim:.4f}, random baseline mean={baseline.mean():.4f}')

        if pooling == 'text_tokens_only':  # primary pooling choice -- save these
            directions[f'text_{organism}'] = direction_text
            directions[f'mm_{organism}'] = direction_mm
            within_organism_cosine[organism] = cos_sim


In [ ]:
for name, direction in directions.items():
    np.save(f'{ARTIFACTS}/direction_{name}.npy', direction)


## Cross-organism convergence check

Do independently-induced organisms converge on the same directions (proposal §4.6/§5.2)? Mirrors Soligo et al.'s original cross-fine-tune convergence test, now crossing induction modality instead of just dataset.

In [ ]:
cos_text_AB = cosine_similarity(directions['text_A'], directions['text_B'])
cos_mm_AB = cosine_similarity(directions['mm_A'], directions['mm_B'])
print(f'direction_text_A vs direction_text_B cosine similarity: {cos_text_AB:.4f}')
print(f'direction_mm_A vs direction_mm_B cosine similarity: {cos_mm_AB:.4f}')

import json

Path(f'{ARTIFACTS}/phase4_direction_summary.json').write_text(json.dumps({
    'within_organism_cosine': within_organism_cosine,  # {'A': cos(text_A, mm_A), 'B': cos(text_B, mm_B)}
    'cross_organism_cosine': {'text_A_vs_text_B': cos_text_AB, 'mm_A_vs_mm_B': cos_mm_AB},
}))
